# Lesson 3: File Explorer

Add `list_files` so the agent can discover what files exist before reading them.

This creates a natural exploration workflow: list → read → understand.

In [ ]:
%pip install openai-agents mermaid-py --upgrade --quiet

<div style='margin-top: 20px; margin-bottom: 20px; text-align: center; display: flex; flex-direction: column; align-items: center;'>
<img src='./images/lesson_3_file_explorer.png' width='600' /> 
</div>

In [ ]:
import os
from getpass import getpass

from agents import Agent, Runner, function_tool

MODEL = "gpt-5.4-mini"

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

In [ ]:
@function_tool
def read_file(path: str) -> str:
    """Read the contents of a file at the given path."""
    try:
        with open(path, "r") as f:
            content = f.read()
        if len(content) > 10000:
            content = content[:10000] + "\n... (truncated)"
        return content
    except FileNotFoundError:
        return f"Error: File not found: {path}"
    except Exception as e:
        return f"Error reading file: {e}"

## Defining the `list_files` Tool

In [ ]:
@function_tool
def list_files(path: str = ".") -> str:
    """List all files and directories at the given path. Directories end with /."""
    try:
        entries = []
        for entry in sorted(os.listdir(path)):
            full = os.path.join(path, entry)
            if os.path.isdir(full):
                entries.append(f"{entry}/")
            else:
                entries.append(entry)
        return "\n".join(entries) if entries else "(empty directory)"
    except FileNotFoundError:
        return f"Error: Directory not found: {path}"
    except Exception as e:
        return f"Error listing files: {e}"

In [ ]:
agent = Agent(
    name="File Explorer",
    instructions="You are a coding assistant that can explore and read files.",
    model=MODEL,
    tools=[read_file, list_files],
)

In [ ]:
result = await Runner.run(agent, "What Python files are in the current directory? Briefly describe what each one does.")
print(result.final_output)

## Summary

- Added `list_files` so the agent can discover files before reading them.
- Tools compose naturally — the agent chains `list_files` → `read_file` on its own.
- Next lesson: add `run_command` to execute shell commands.